# 1D Pegasus Phase Diagram

This notebook keeps the 1D data loading and thermodynamic phase classification from
`phase_diagram_1d_simple.ipynb`, then applies the publication-style plotting treatment
from `phase_diagram_2d_old.ipynb`: LaTeX fonts, fuzzy phase boundaries, a beta-comparison
inset, and TikZ operating-mode illustrations.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import PolyCollection
from matplotlib.colors import ListedColormap
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from scipy.spatial import Voronoi, cKDTree

try:
    from pdf2image import convert_from_path
except ImportError:
    convert_from_path = None


PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "plots" else Path.cwd()
PLOTS_DIR = PROJECT_ROOT / "plots"
DATA_PATH = PROJECT_ROOT / "data/results/phase_diagrams_1d_pegasus_unif_small_h"
TIKZ_DIR = PLOTS_DIR / "tikz"
OUTPUT_DIR = PLOTS_DIR / "1d_chain"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CHAIN_LENGTH = 300
ANNEALING_TIMES = (100.0, 2000.0)
ANNEALING_TIME = ANNEALING_TIMES[-1]

MIN_BETA = 2.0
MAX_BETA = 6.0
MIN_AP = 0.1
MAX_AP = 0.65
PANEL_LIMITS = {
    100.0: {"x": (0.1, 0.65), "y": (3.0, 5.0)},
    2000.0: {"x": (0.1, 0.65), "y": (3.0, 5.0)},
}

# Manual phase edits applied before undetermined-point filling and isolated-point cleanup.
# Each override uses x=(s_min, s_max), y=(beta1_min, beta1_max), and phase=<integer key from PHASE_LABELS>.
# Overrides only change existing finite grid cells inside the rectangle; they never create new data points.
PHASE_OVERRIDES = {
    100.0: [
        {"x": (0.45, 0.55), "y": (4.5, 5.0), "phase": 4},
    ],
    2000.0: [
        {"x": (0.10, 0.40), "y": (4.5, 5.2), "phase": 4},
    ],
}

INEQUALITY_TOL = 1e-4
K = 2
CLEAN_ISOLATED_PHASE_POINTS = True
FILL_UNDETERMINED_PHASE_POINTS = True
DRAW_CELL_BOUNDARY = False
DRAW_SAMPLE_POINTS = True
SHOW_FUZZY_BOUNDARY = False
DRAW_INTERPOLATED_PHASE_BOUNDARY = True
INTERPOLATED_PHASE_BOUNDARY_GRID_SIZE = 900
INTERPOLATED_PHASE_BOUNDARY_EDGE_COLOR = "black"
INTERPOLATED_PHASE_BOUNDARY_INTERIOR_COLOR = "white"
INTERPOLATED_PHASE_BOUNDARY_EDGE_LINEWIDTH = 2.0
INTERPOLATED_PHASE_BOUNDARY_INTERIOR_LINEWIDTH = 10.0
INTERPOLATED_PHASE_BOUNDARY_SMOOTHING = 50.0
FUZZY_BOUNDARY_COLOR = "black"
FUZZY_BOUNDARY_WIDTHS = (5.0, 2.4, 0.8)
FUZZY_BOUNDARY_ALPHAS = (0.07, 0.14, 0.28)
FUZZY_BOUNDARY_GRID_SIZE = 800
FUZZY_BOUNDARY_SMOOTHING = 3.0
UNCERTAIN_BOUNDARY_EXCLUDED_REGIONS = []

PHASE_LABELS = {
    0: "Undetermined",
    1: "Refrigerator [R]",
    2: "Engine [E]",
    3: "Accelerator [A]",
    4: "Heater [H]",
}

PHASE_SHORT_LABELS = {
    0: "U",
    1: "R",
    2: "E",
    3: "A",
    4: "H",
}

PHASE_COLORS = {
    0: "lightgray",
    1: "blue",
    2: "red",
    3: "green",
    4: "orange",
}

BETA_COMPARISON_LABELS = {
    0: r"$\beta_1 \approx \beta_2$",
    1: r"$\beta_1 < \beta_2$",
    2: r"$\beta_1 > \beta_2$",
}

BETA_COMPARISON_COLORS = {
    0: "0.65",
    1: "lightblue",
    2: "lightcoral",
}

PHASE_TIKZ_FILES = {
    1: "refrigerator.pdf",
    2: "engine.pdf",
    3: "accelerator.pdf",
    4: "heater.pdf",
}


def latex_plot(scale=1, fontsize=12):
    """Apply the LaTeX plot styling used by the 2D phase-diagram notebook."""
    fig_width_pt = 246.0
    inches_per_pt = 1.0 / 72.27
    golden_mean = (np.sqrt(5.0) - 1.0) / 2.0
    fig_width = fig_width_pt * inches_per_pt * scale
    fig_height = fig_width * golden_mean
    mpl.rcParams.update(
        {
            "pgf.texsystem": "pdflatex",
            "text.usetex": True,
            "font.family": "serif",
            "font.serif": [],
            "font.sans-serif": [],
            "font.monospace": [],
            "axes.labelsize": fontsize,
            "font.size": fontsize,
            "legend.fontsize": fontsize - 1,
            "xtick.labelsize": fontsize - 1,
            "ytick.labelsize": fontsize - 1,
            "figure.figsize": [fig_width, fig_height],
        }
    )


In [ ]:
def as_dict(obj):
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "item"):
        maybe = obj.item()
        if isinstance(maybe, dict):
            return maybe
    raise TypeError(f"Expected a dict-like object, got {type(obj)}")


def load_data():
    beta_raw = as_dict(np.load(DATA_PATH / f"betas2_{CHAIN_LENGTH}.pkl", allow_pickle=True))
    q_raw = as_dict(np.load(DATA_PATH / f"Q_{CHAIN_LENGTH}.pkl", allow_pickle=True))
    return beta_raw, q_raw


def g_func(x):
    x = np.clip(x, -0.999999, 0.999999)
    return x * np.arctanh(x)


def classify(beta1, beta2, dE1, dE2, work, tol=INEQUALITY_TOL):
    if min(abs(dE1), abs(dE2), abs(work), abs(beta1 - beta2)) <= tol:
        return 0

    if beta1 < beta2:
        if dE1 > tol and dE2 < -tol and work > tol:
            return 1
        if dE1 < -tol and dE2 > tol and work < -tol:
            return 2
        if dE1 < -tol and dE2 > tol and work > tol:
            return 3
        if dE1 > tol and dE2 > tol and work > tol:
            return 4
    else:
        if dE1 < -tol and dE2 > tol and work > tol:
            return 1
        if dE1 > tol and dE2 < -tol and work < -tol:
            return 2
        if dE1 > tol and dE2 < -tol and work > tol:
            return 3
        if dE1 > tol and dE2 > tol and work > tol:
            return 4

    return 0


def iter_filtered_records(beta_raw, q_raw=None, annealing_time=ANNEALING_TIME):
    annealing_time = float(annealing_time)
    for key, beta2_raw in beta_raw.items():
        if q_raw is not None and key not in q_raw:
            continue

        chain_length, key_annealing_time, ap, beta1 = key
        chain_length = int(chain_length)
        key_annealing_time = float(key_annealing_time)
        ap = float(ap)
        beta1 = float(beta1)

        if chain_length != CHAIN_LENGTH or key_annealing_time != annealing_time:
            continue
        if not (MIN_BETA <= beta1 <= MAX_BETA and MIN_AP <= ap <= MAX_AP):
            continue

        beta2 = -float(beta2_raw)
        if not np.isfinite(beta2) or abs(beta2) <= 1e-10:
            continue

        if q_raw is None:
            yield key, ap, beta1, beta2, None
        else:
            q_mean, q_var = q_raw[key]
            yield key, ap, beta1, beta2, (float(q_mean), float(q_var))


def build_phase_grid(beta_raw, q_raw, annealing_time=ANNEALING_TIME):
    records = []
    for _, ap, beta1, beta2, q_values in iter_filtered_records(beta_raw, q_raw, annealing_time):
        q_mean, q_var = q_values
        ratio = q_mean / np.sqrt(q_var + q_mean**2)
        dE1 = q_mean
        dE2 = (2 / beta2) * g_func(ratio) - (beta1 / beta2) * q_mean
        work = dE1 + dE2
        phase = classify(beta1, beta2, dE1, dE2, work)
        records.append((beta1, ap, phase))

    if not records:
        raise ValueError(f"No data points survived filtering for t={annealing_time:g} us.")

    beta_values = np.array(sorted({row[0] for row in records}), dtype=float)
    ap_values = np.array(sorted({row[1] for row in records}), dtype=float)
    phases = np.full((len(ap_values), len(beta_values)), np.nan)
    beta_index = {value: idx for idx, value in enumerate(beta_values)}
    ap_index = {value: idx for idx, value in enumerate(ap_values)}

    for beta1, ap, phase in records:
        phases[ap_index[ap], beta_index[beta1]] = phase

    return beta_values, ap_values, phases



def apply_phase_overrides(phases, beta_values, ap_values, overrides):
    overridden = phases.copy()
    applied = []

    for override in overrides:
        x_min, x_max = sorted(override["x"])
        y_min, y_max = sorted(override["y"])
        phase = int(override["phase"])
        if phase not in PHASE_LABELS:
            raise ValueError(f"Unknown phase override value: {phase}")

        row_mask = (ap_values >= x_min) & (ap_values <= x_max)
        col_mask = (beta_values >= y_min) & (beta_values <= y_max)
        existing_data_mask = np.isfinite(phases)
        override_mask = existing_data_mask & row_mask[:, None] & col_mask[None, :]
        rows, cols = np.nonzero(override_mask)

        for row, col in zip(rows, cols):
            old_phase = overridden[row, col]
            overridden[row, col] = phase
            applied.append((ap_values[row], beta_values[col], old_phase, phase))

    return overridden, applied

def clean_isolated_points(phases, beta_values, ap_values):
    cleaned = phases.copy()
    replacements = []

    for row in range(phases.shape[0]):
        for col in range(phases.shape[1]):
            phase = phases[row, col]
            if not np.isfinite(phase) or int(phase) == 0:
                continue

            neighbors = []
            for drow in (-1, 0, 1):
                for dcol in (-1, 0, 1):
                    if drow == 0 and dcol == 0:
                        continue
                    r = row + drow
                    c = col + dcol
                    if 0 <= r < phases.shape[0] and 0 <= c < phases.shape[1]:
                        neighbor = phases[r, c]
                        if np.isfinite(neighbor) and int(neighbor) != 0:
                            neighbors.append(int(neighbor))

            if not neighbors:
                continue

            phase = int(phase)
            same_count = sum(neighbor == phase for neighbor in neighbors)
            if same_count > K:
                continue

            candidates = {neighbor: neighbors.count(neighbor) for neighbor in set(neighbors) if neighbor != phase}
            if not candidates:
                continue

            new_phase, new_count = max(candidates.items(), key=lambda item: item[1])
            if list(candidates.values()).count(new_count) == 1 and new_count > same_count:
                cleaned[row, col] = new_phase
                replacements.append((ap_values[row], beta_values[col], phase, new_phase, same_count))

    return cleaned, replacements



def fill_undetermined_points(phases):
    filled = phases.copy()
    replacements = []
    max_passes = filled.size

    for _ in range(max_passes):
        pass_replacements = []
        for row in range(filled.shape[0]):
            for col in range(filled.shape[1]):
                phase = filled[row, col]
                if not np.isfinite(phase) or int(phase) != 0:
                    continue

                neighbors = []
                for drow in (-1, 0, 1):
                    for dcol in (-1, 0, 1):
                        if drow == 0 and dcol == 0:
                            continue
                        r = row + drow
                        c = col + dcol
                        if 0 <= r < filled.shape[0] and 0 <= c < filled.shape[1]:
                            neighbor = filled[r, c]
                            if np.isfinite(neighbor) and int(neighbor) != 0:
                                neighbors.append(int(neighbor))

                if not neighbors:
                    continue

                counts = {neighbor: neighbors.count(neighbor) for neighbor in set(neighbors)}
                new_phase = max(counts.items(), key=lambda item: (item[1], -item[0]))[0]
                pass_replacements.append((row, col, new_phase))

        if not pass_replacements:
            break

        for row, col, new_phase in pass_replacements:
            filled[row, col] = new_phase
            replacements.append((row, col, new_phase))

    return filled, replacements

def print_phase_counts(phases, title):
    values = phases[np.isfinite(phases)].astype(int)
    print(title)
    for phase in sorted(np.unique(values)):
        print(f"  {PHASE_LABELS[phase]}: {np.sum(values == phase)}")


def build_beta_comparison_points(beta_raw, annealing_time=ANNEALING_TIME):
    rows = []
    for _, ap, beta1, beta2, _ in iter_filtered_records(beta_raw, annealing_time=annealing_time):
        if abs(beta1 - beta2) <= INEQUALITY_TOL:
            comparison = 0
        elif beta1 < beta2:
            comparison = 1
        else:
            comparison = 2
        rows.append((ap, beta1, comparison))

    if len(rows) < 4:
        raise ValueError(f"At least 4 beta-comparison points are needed for t={annealing_time:g} us.")

    data = np.array(sorted(rows, key=lambda row: (row[0], row[1])), dtype=float)
    return data[:, :2], data[:, 2].astype(int)


In [ ]:
def voronoi_finite_polygons_2d(vor, radius=None):
    if vor.points.shape[1] != 2:
        raise ValueError("Voronoi input must be 2D")

    new_regions = []
    new_vertices = vor.vertices.tolist()
    center = vor.points.mean(axis=0)
    if radius is None:
        radius = np.ptp(vor.points, axis=0).max() * 2

    all_ridges = {}
    for (p1, p2), (v1, v2) in zip(vor.ridge_points, vor.ridge_vertices):
        all_ridges.setdefault(p1, []).append((p2, v1, v2))
        all_ridges.setdefault(p2, []).append((p1, v1, v2))

    for p1, region_idx in enumerate(vor.point_region):
        vertices = vor.regions[region_idx]
        if all(v >= 0 for v in vertices):
            new_regions.append(vertices)
            continue

        new_region = [v for v in vertices if v >= 0]
        for p2, v1, v2 in all_ridges[p1]:
            if v2 < 0:
                v1, v2 = v2, v1
            if v1 >= 0:
                continue

            tangent = vor.points[p2] - vor.points[p1]
            tangent /= np.linalg.norm(tangent)
            normal = np.array([-tangent[1], tangent[0]])
            midpoint = vor.points[[p1, p2]].mean(axis=0)
            direction = np.sign(np.dot(midpoint - center, normal)) * normal
            far_point = vor.vertices[v2] + direction * radius
            new_region.append(len(new_vertices))
            new_vertices.append(far_point.tolist())

        vs = np.asarray([new_vertices[v] for v in new_region])
        angles = np.arctan2(vs[:, 1] - vs[:, 1].mean(), vs[:, 0] - vs[:, 0].mean())
        new_regions.append(np.array(new_region)[np.argsort(angles)].tolist())

    return new_regions, np.asarray(new_vertices)


def clip_polygon_to_box(polygon, x_min, x_max, y_min, y_max):
    def clip(points, inside, intersect):
        clipped = []
        previous = points[-1]
        previous_inside = inside(previous)
        for current in points:
            current_inside = inside(current)
            if current_inside:
                if not previous_inside:
                    clipped.append(intersect(previous, current))
                clipped.append(current)
            elif previous_inside:
                clipped.append(intersect(previous, current))
            previous, previous_inside = current, current_inside
        return clipped

    eps = 1e-12
    points = [np.asarray(point, dtype=float) for point in polygon]
    for inside, intersect in [
        (lambda p: p[0] >= x_min, lambda a, b: a + (b - a) * ((x_min - a[0]) / (b[0] - a[0] + eps))),
        (lambda p: p[0] <= x_max, lambda a, b: a + (b - a) * ((x_max - a[0]) / (b[0] - a[0] + eps))),
        (lambda p: p[1] >= y_min, lambda a, b: a + (b - a) * ((y_min - a[1]) / (b[1] - a[1] + eps))),
        (lambda p: p[1] <= y_max, lambda a, b: a + (b - a) * ((y_max - a[1]) / (b[1] - a[1] + eps))),
    ]:
        if not points:
            break
        points = clip(points, inside, intersect)
    return np.asarray(points)


def points_in_rectangles(x, y, rectangles):
    x = np.asarray(x)
    y = np.asarray(y)
    mask = np.zeros(np.broadcast_shapes(x.shape, y.shape), dtype=bool)
    for (x1, x2), (y1, y2) in rectangles:
        x_min, x_max = sorted((x1, x2))
        y_min, y_max = sorted((y1, y2))
        mask |= (x_min <= x) & (x <= x_max) & (y_min <= y) & (y <= y_max)
    return mask


def build_voronoi_polygons(points, values, x_limits, y_limits):
    if len(points) < 4:
        raise ValueError("At least 4 points are needed to draw Voronoi cells.")

    vor = Voronoi(points)
    regions, vertices = voronoi_finite_polygons_2d(vor)
    polygons = []
    polygon_values = []
    for region, value in zip(regions, values):
        polygon = clip_polygon_to_box(vertices[region], x_limits[0], x_limits[1], y_limits[0], y_limits[1])
        if len(polygon) >= 3:
            polygons.append(polygon)
            polygon_values.append(int(value))
    return polygons, np.array(polygon_values, dtype=int), vor


def add_fuzzy_phase_boundary(ax, points, phases, vor):
    if not SHOW_FUZZY_BOUNDARY:
        return False

    uncertain_mask = np.zeros(len(points), dtype=bool)
    for p1, p2 in vor.ridge_points:
        if phases[p1] != phases[p2]:
            uncertain_mask[p1] = True
            uncertain_mask[p2] = True

    uncertain_mask[points_in_rectangles(points[:, 0], points[:, 1], UNCERTAIN_BOUNDARY_EXCLUDED_REGIONS)] = False
    if not np.any(uncertain_mask):
        return False

    ap_grid = np.linspace(MIN_AP, MAX_AP, FUZZY_BOUNDARY_GRID_SIZE)
    beta_grid = np.linspace(MIN_BETA, MAX_BETA, FUZZY_BOUNDARY_GRID_SIZE)
    ap_surface, beta_surface = np.meshgrid(ap_grid, beta_grid)
    surface_points = np.column_stack([ap_surface.ravel(), beta_surface.ravel()])
    _, nearest_indices = cKDTree(points).query(surface_points, k=1)
    uncertain_surface = uncertain_mask[nearest_indices].reshape(ap_surface.shape).astype(float)
    uncertain_surface = gaussian_filter(uncertain_surface, sigma=FUZZY_BOUNDARY_SMOOTHING)
    excluded_surface = points_in_rectangles(ap_surface, beta_surface, UNCERTAIN_BOUNDARY_EXCLUDED_REGIONS)
    uncertain_surface[excluded_surface] = np.nan

    for fuzzy_width, fuzzy_alpha in zip(FUZZY_BOUNDARY_WIDTHS, FUZZY_BOUNDARY_ALPHAS):
        ax.contour(
            ap_surface,
            beta_surface,
            uncertain_surface,
            levels=[0.5],
            colors=FUZZY_BOUNDARY_COLOR,
            linewidths=fuzzy_width,
            alpha=fuzzy_alpha,
        )
    return True



def add_interpolated_phase_boundaries(ax, points, phases, x_limits, y_limits):
    if not DRAW_INTERPOLATED_PHASE_BOUNDARY:
        return False

    present_phases = sorted(set(int(phase) for phase in phases if np.isfinite(phase)))
    if len(present_phases) < 2:
        return False

    ap_grid = np.linspace(x_limits[0], x_limits[1], INTERPOLATED_PHASE_BOUNDARY_GRID_SIZE)
    beta_grid = np.linspace(y_limits[0], y_limits[1], INTERPOLATED_PHASE_BOUNDARY_GRID_SIZE)
    ap_surface, beta_surface = np.meshgrid(ap_grid, beta_grid)
    surface_points = np.column_stack([ap_surface.ravel(), beta_surface.ravel()])
    _, nearest_indices = cKDTree(points).query(surface_points, k=1)
    phase_surface = phases[nearest_indices].reshape(ap_surface.shape).astype(float)
    smooth_phase_surface = gaussian_filter(phase_surface, sigma=INTERPOLATED_PHASE_BOUNDARY_SMOOTHING)

    levels = [phase + 0.5 for phase in range(min(present_phases), max(present_phases))]
    if not levels:
        return False

    edge_contour = ax.contour(
        ap_surface,
        beta_surface,
        smooth_phase_surface,
        levels=levels,
        colors=INTERPOLATED_PHASE_BOUNDARY_EDGE_COLOR,
        linewidths=INTERPOLATED_PHASE_BOUNDARY_EDGE_LINEWIDTH,
        alpha=1.0,
        zorder=8,
    )
    interior_contour = ax.contour(
        ap_surface,
        beta_surface,
        smooth_phase_surface,
        levels=levels,
        colors=INTERPOLATED_PHASE_BOUNDARY_INTERIOR_COLOR,
        linewidths=INTERPOLATED_PHASE_BOUNDARY_INTERIOR_LINEWIDTH,
        alpha=1.0,
        zorder=9,
    )
    return any(len(path.vertices) > 0 for path in edge_contour.get_paths()) or any(
        len(path.vertices) > 0 for path in interior_contour.get_paths()
    )

def pdf_to_image_with_background(pdf_path, dpi=300, bg_alpha=0.9):
    if convert_from_path is None:
        raise ImportError("pdf2image is required to render TikZ PDF insets.")

    pages = convert_from_path(str(pdf_path), dpi=dpi)
    img = pages[0].convert("RGBA")
    img_array = np.array(img)
    white_threshold = 250
    white_mask = (
        (img_array[:, :, 0] > white_threshold)
        & (img_array[:, :, 1] > white_threshold)
        & (img_array[:, :, 2] > white_threshold)
    )
    img_array[white_mask, 3] = int(bg_alpha * 255)
    return img_array


def representative_phase_positions(points, phases):
    positions = {}
    for phase in sorted(set(phases)):
        if phase == 0:
            continue
        phase_points = points[phases == phase]
        if len(phase_points) == 0:
            continue
        center = np.median(phase_points, axis=0)
        nearest = phase_points[np.argmin(np.linalg.norm(phase_points - center, axis=1))]
        positions[int(phase)] = (float(nearest[0]), float(nearest[1]))
    return positions


def add_mode_insets(ax, points, phases, zoom=0.105, bg_alpha=0.92):
    positions = representative_phase_positions(points, phases)
    added = []
    for phase, position in positions.items():
        pdf_name = PHASE_TIKZ_FILES.get(phase)
        if pdf_name is None:
            continue
        pdf_path = TIKZ_DIR / pdf_name
        if not pdf_path.exists():
            print(f"Skipping {PHASE_LABELS[phase]} inset; missing {pdf_path}")
            continue
        try:
            image = pdf_to_image_with_background(pdf_path, dpi=300, bg_alpha=bg_alpha)
        except Exception as exc:
            print(f"Skipping {PHASE_LABELS[phase]} inset; could not load {pdf_path}: {exc}")
            continue
        imagebox = OffsetImage(image, zoom=zoom)
        inset = AnnotationBbox(
            imagebox,
            position,
            frameon=True,
            box_alignment=(0.5, 0.5),
            bboxprops={"boxstyle": "round,pad=0.12", "facecolor": "white", "edgecolor": "0.25", "alpha": 0.88},
        )
        ax.add_artist(inset)
        added.append(PHASE_LABELS[phase])
    return added


In [ ]:
beta_raw, q_raw = load_data()
slice_data = {}

print(f"Loaded beta entries: {len(beta_raw)}")
print(f"Loaded Q entries: {len(q_raw)}")

for annealing_time in ANNEALING_TIMES:
    beta_values, ap_values, phase_grid = build_phase_grid(beta_raw, q_raw, annealing_time)
    phase_grid, manual_overrides = apply_phase_overrides(
        phase_grid,
        beta_values,
        ap_values,
        PHASE_OVERRIDES.get(float(annealing_time), []),
    )

    if FILL_UNDETERMINED_PHASE_POINTS:
        phase_grid, undetermined_replacements = fill_undetermined_points(phase_grid)
    else:
        undetermined_replacements = []

    if CLEAN_ISOLATED_PHASE_POINTS:
        phase_grid, isolated_replacements = clean_isolated_points(phase_grid, beta_values, ap_values)
        phase_counts_title = f"Phase counts after cleanup, t={annealing_time:g} us:"
    else:
        isolated_replacements = []
        phase_counts_title = f"Phase counts without cleanup, t={annealing_time:g} us:"

    comparison_points, comparison_values = build_beta_comparison_points(beta_raw, annealing_time)
    slice_data[annealing_time] = {
        "beta_values": beta_values,
        "ap_values": ap_values,
        "phase_grid": phase_grid,
        "comparison_points": comparison_points,
        "comparison_values": comparison_values,
        "manual_overrides": manual_overrides,
        "undetermined_replacements": undetermined_replacements,
        "isolated_replacements": isolated_replacements,
    }

    print()
    print(f"t={annealing_time:g} us")
    print(f"  Plotted grid shape: {phase_grid.shape[0]} ap values x {phase_grid.shape[1]} beta1 values")
    print(f"  beta1 range: [{beta_values.min():g}, {beta_values.max():g}]")
    print(f"  ap range: [{ap_values.min():g}, {ap_values.max():g}]")
    print_phase_counts(phase_grid, phase_counts_title)
    print(f"  Manual phase overrides: {len(manual_overrides)}")
    print(f"  Filled undetermined points: {len(undetermined_replacements)}")
    print(f"  Isolated-point replacements: {len(isolated_replacements)}")
    print(f"  Beta-comparison points: {len(comparison_points)}")
    for value in sorted(set(comparison_values)):
        print(f"    {BETA_COMPARISON_LABELS[value]}: {np.sum(comparison_values == value)}")


In [ ]:
def plot_phase_panel(ax, annealing_time, data):
    panel_limits = PANEL_LIMITS[float(annealing_time)]
    x_limits = panel_limits["x"]
    y_limits = panel_limits["y"]

    beta_values = data["beta_values"]
    ap_values = data["ap_values"]
    phase_grid = data["phase_grid"]
    comparison_points = data["comparison_points"]
    comparison_values = data["comparison_values"]

    beta_mesh, ap_mesh = np.meshgrid(beta_values, ap_values)
    valid = np.isfinite(phase_grid)
    points = np.column_stack([ap_mesh[valid], beta_mesh[valid]])
    phases = phase_grid[valid].astype(int)

    phase_polygons, polygon_phases, vor = build_voronoi_polygons(
        points,
        phases,
        x_limits=x_limits,
        y_limits=y_limits,
    )

    cells = PolyCollection(
        phase_polygons,
        facecolors=[PHASE_COLORS[phase] for phase in polygon_phases],
        edgecolors="black" if DRAW_CELL_BOUNDARY else "none",
        linewidths=0.25 if DRAW_CELL_BOUNDARY else 0.0,
        alpha=0.72,
    )
    ax.add_collection(cells)

    fuzzy_added = add_fuzzy_phase_boundary(ax, points, phases, vor)

    if DRAW_SAMPLE_POINTS:
        ax.scatter(points[:, 0], points[:, 1], s=6, c="black", alpha=0.24, linewidths=0, zorder=4)

    interpolated_boundary_added = add_interpolated_phase_boundaries(ax, points, phases, x_limits, y_limits)

    grid_resolution = 320
    ap_grid = np.linspace(x_limits[0], x_limits[1], grid_resolution)
    beta_grid = np.linspace(y_limits[0], y_limits[1], grid_resolution)
    ap_surface, beta_surface = np.meshgrid(ap_grid, beta_grid)
    beta_comp_interpolated = griddata(
        comparison_points,
        comparison_values,
        (ap_surface, beta_surface),
        method="nearest",
    ).astype(int)
    beta_comp_interpolated = np.clip(beta_comp_interpolated, 0, 2)

    ax_inset = inset_axes(
        ax,
        width="31%",
        height="31%",
        loc="lower right",
        borderpad=0.9,
    )
    beta_comp_cmap = ListedColormap(
        [BETA_COMPARISON_COLORS[0], BETA_COMPARISON_COLORS[1], BETA_COMPARISON_COLORS[2]]
    )
    ax_inset.imshow(
        beta_comp_interpolated,
        extent=[x_limits[0], x_limits[1], y_limits[0], y_limits[1]],
        cmap=beta_comp_cmap,
        origin="lower",
        aspect="auto",
        vmin=0,
        vmax=2,
    )
    ax_inset.set_xlabel(r"$s$", fontsize=8)
    ax_inset.set_ylabel(r"$\beta_1$", fontsize=8)
    ax_inset.tick_params(axis="both", labelsize=7, length=2)
    ax_inset.set_xlim(*x_limits)
    ax_inset.set_ylim(*y_limits)
    ax_inset.set_facecolor("white")
    for spine in ax_inset.spines.values():
        spine.set_color("0.2")
        spine.set_linewidth(0.8)
    ax_inset.legend(
        handles=[
            Patch(facecolor=BETA_COMPARISON_COLORS[1], edgecolor="black", label=BETA_COMPARISON_LABELS[1]),
            Patch(facecolor=BETA_COMPARISON_COLORS[2], edgecolor="black", label=BETA_COMPARISON_LABELS[2]),
        ],
        loc="upper left",
        fontsize=5.8,
        framealpha=0.92,
        handlelength=0.8,
        borderpad=0.2,
    )

    added_insets = add_mode_insets(ax, points, phases, zoom=0.08)

    ax.set_xlabel(r"Reverse annealing parameter $s$")
    ax.set_ylabel(r"Initial state inverse temperature $\beta_1$")
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    ax.set_box_aspect(1)
    ax.grid(True, alpha=0.12)
    ax.set_title(rf"$t={annealing_time:g}\,\mu\mathrm{{s}}$")

    return sorted(set(phases)), fuzzy_added, interpolated_boundary_added, added_insets


latex_plot(scale=2.3, fontsize=13)
fig, axes = plt.subplots(1, 2, figsize=(10.8, 5.4), sharex=True, sharey=False)

all_present_phases = set()
fuzzy_added_anywhere = False
interpolated_boundary_added_anywhere = False
inset_summary = {}
for panel_index, annealing_time in enumerate(ANNEALING_TIMES):
    present_phases, fuzzy_added, interpolated_boundary_added, added_insets = plot_phase_panel(
        axes[panel_index],
        annealing_time,
        slice_data[annealing_time],
    )
    all_present_phases.update(present_phases)
    fuzzy_added_anywhere = fuzzy_added_anywhere or fuzzy_added
    interpolated_boundary_added_anywhere = interpolated_boundary_added_anywhere or interpolated_boundary_added
    inset_summary[annealing_time] = added_insets

handles = [
    Patch(facecolor=PHASE_COLORS[phase], label=PHASE_LABELS[phase])
    for phase in sorted(all_present_phases)
]
# if fuzzy_added_anywhere:
    # handles.append(Patch(facecolor="none", edgecolor=FUZZY_BOUNDARY_COLOR, linewidth=2.0, label="Fuzzy phase boundary"))
# if interpolated_boundary_added_anywhere:
    # handles.append(Patch(facecolor="none", edgecolor=INTERPOLATED_PHASE_BOUNDARY_EDGE_COLOR, linewidth=INTERPOLATED_PHASE_BOUNDARY_EDGE_LINEWIDTH, label="Interpolated phase boundary"))

fig.legend(handles=handles, loc="lower center", ncol=min(len(handles), 5), framealpha=0.94, bbox_to_anchor=(0.5, -0.015))
# fig.tight_layout(rect=(0, 0.08, 1, 0.96))

output_path = OUTPUT_DIR / "phase_diagram_1d_at_100_and_2000.pdf"
fig.savefig(output_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Saved {output_path}")
for annealing_time, added_insets in inset_summary.items():
    print(f"t={annealing_time:g} us TikZ insets:", ", ".join(added_insets) if added_insets else "none")
